# Call Center Confidence Model (Improved with More Diverse Synthetic Data)

**Goal**: Predict confidence level (high / medium / low) → escalation decision

**Improvements**:
- Much more diverse synthetic data (~400 examples)
- Variation pools → no duplicates, better generalization
- Balanced classes
- Using bert-base-uncased

Date: Jan 2026

In [1]:
!pip install -q transformers datasets torch scikit-learn pandas tqdm

In [2]:
import pandas as pd
import numpy as np
import random
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments
)
from datasets import Dataset as HFDataset

# Prevent TPU probing (just in case)
import os
os.environ["PJRT_DEVICE"] = "cpu"

## 1. Create larger & more diverse synthetic dataset

In [5]:
np.random.seed(42)
random.seed(42)
data = []

def add_example(customer_text, agent_text, confidence_label, score):
    text = f"Customer: {customer_text}\nAgent: {agent_text}"
    data.append({"text": text, "label": confidence_label, "score": score})

# === Variation pools ===

# High confidence - clear resolution, helpful, professional/casual mix
high_cust = [
    "My package still hasn't arrived.", "Order is late, where is it?", "Haven't received my parcel yet.",
    "Internet keeps dropping every few minutes.", "Connection drops constantly.", "WiFi is unstable all day.",
    "The bill looks wrong, I was overcharged.", "Extra charge on my last bill.", "Billing seems incorrect."
]
high_agent = [
    "Checked it — delayed due to weather. Expedited now + $10 credit.", "Updated delivery to tomorrow, added compensation.",
    "Ran diagnostics — fixed the node, connection is stable now.", "You're right, billing error. Refund $18.50 processed.",
    "No worries bro, glad it's sorted! 😎", "Sweet! Happy to help dude.", "Awesome, cheers!", "Anytime! Take care.",
    "All good now — thanks for your patience!"
]

# Medium confidence - ongoing, need more info, partial progress
medium_cust = [
    "App crashes when I try to pay.", "Payment crashes the app.", "Can't complete transaction, app freezes.",
    "Want to change my plan.", "Need to upgrade/downgrade my package.", "Looking to switch to a different plan.",
    "SIM not working abroad.", "Roaming not working overseas."
]
medium_agent = [
    "Which device and OS version are you using?", "Can you tell me your phone model?",
    "What error message do you see?", "Let me check available plans... what features do you need?",
    "Roaming looks enabled — try restarting the phone?", "Need your location/country to check settings."
]

# Low confidence - unhelpful, off-topic, confused, looping
low_cust = [
    "It's working now actually.", "All good now, thanks.", "Problem solved.", "Thanks, fixed now.",
    "This is the third time! Nothing works!", "You're useless, give me a manager!"
]
low_agent = [
    "Oh... so everything is fine? Anything is possible...", "Great! Wait... what was the issue again?",
    "Perfect! Want our new insurance add-on?", "I see... should I open a ticket or cancel?",
    "I'm sorry... let me see what I can do... (same answer)", "I understand you're upset... can't transfer right now."
]

In [6]:
def generate_category(cust_pool, agent_pool, label, score_range, count, max_attempts=5000):
    used = set()
    attempts = 0
    
    print(f"Generating {count} examples for {label}...")
    
    while len(used) < count and attempts < max_attempts:
        attempts += 1
        cust = random.choice(cust_pool)
        agent = random.choice(agent_pool)
        text = f"Customer: {cust}\nAgent: {agent}"
        
        if text not in used:
            used.add(text)
            score = round(random.uniform(*score_range), 2)
            add_example(cust, agent, label, score)
        
        if attempts % 500 == 0:
            print(f"  Attempts: {attempts} | Unique found: {len(used)}/{count}")
    
    if attempts >= max_attempts:
        print(f"Warning: Reached max attempts ({max_attempts}). Only got {len(used)}/{count} unique examples")
    
    print(f"Done → {len(used)} unique {label} examples\n")

In [7]:
generate_category(high_cust, high_agent, "high",   (0.80, 0.96), 80)
generate_category(medium_cust, medium_agent, "medium", (0.45, 0.70), 60)
generate_category(low_cust, low_agent, "low",   (0.10, 0.40), 80)

Generating 80 examples for high...
Done → 80 unique high examples

Generating 60 examples for medium...
  Attempts: 500 | Unique found: 48/60
  Attempts: 1000 | Unique found: 48/60
  Attempts: 1500 | Unique found: 48/60
  Attempts: 2000 | Unique found: 48/60
  Attempts: 2500 | Unique found: 48/60
  Attempts: 3000 | Unique found: 48/60
  Attempts: 3500 | Unique found: 48/60
  Attempts: 4000 | Unique found: 48/60
  Attempts: 4500 | Unique found: 48/60
  Attempts: 5000 | Unique found: 48/60
Done → 48 unique medium examples

Generating 80 examples for low...
  Attempts: 500 | Unique found: 36/80
  Attempts: 1000 | Unique found: 36/80
  Attempts: 1500 | Unique found: 36/80
  Attempts: 2000 | Unique found: 36/80
  Attempts: 2500 | Unique found: 36/80
  Attempts: 3000 | Unique found: 36/80
  Attempts: 3500 | Unique found: 36/80
  Attempts: 4000 | Unique found: 36/80
  Attempts: 4500 | Unique found: 36/80
  Attempts: 5000 | Unique found: 36/80
Done → 36 unique low examples



In [8]:
df = pd.DataFrame(data)
print(f"Total unique examples: {len(df)}")
print(df['label'].value_counts())
print(f"Duplicates remaining: {df['text'].duplicated().sum()}")
df.sample(8)

Total unique examples: 164
label
high      80
medium    48
low       36
Name: count, dtype: int64
Duplicates remaining: 0


,text,label,score
135,Customer: It's working now actually.\nAgent: O...,low,0.27
115,"Customer: Can't complete transaction, app free...",medium,0.46
131,Customer: Problem solved.\nAgent: Oh... so eve...,low,0.31
55,Customer: My package still hasn't arrived.\nAg...,high,0.88
95,Customer: Looking to switch to a different pla...,medium,0.60
29,Customer: My package still hasn't arrived.\nAg...,high,0.82
157,"Customer: All good now, thanks.\nAgent: Great!...",low,0.34
51,"Customer: Order is late, where is it?\nAgent: ...",high,0.93


## 2. Prepare data

In [9]:
label2id = {"low": 0, "medium": 1, "high": 2}
id2label = {v: k for k, v in label2id.items()}

df['label_id'] = df['label'].map(label2id)

train_df, test_df = train_test_split(
    df, test_size=0.25, stratify=df['label_id'], random_state=42
)

print(f"Train: {len(train_df)}   Test: {len(test_df)}")

Train: 123   Test: 41


In [10]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=256)

train_dataset = HFDataset.from_pandas(train_df).map(tokenize_function, batched=True)
test_dataset  = HFDataset.from_pandas(test_df).map(tokenize_function, batched=True)

train_dataset = train_dataset.rename_column("label_id", "labels")
test_dataset  = test_dataset.rename_column("label_id", "labels")

train_dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
test_dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Map:   0%|          | 0/123 [00:00<?, ? examples/s]

Map:   0%|          | 0/41 [00:00<?, ? examples/s]

## 3. Fine-tune BERT

In [12]:
model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=3,
    id2label=id2label,
    label2id=label2id
)

training_args = TrainingArguments(
    output_dir="./confidence_bert_results",
    num_train_epochs=5,                       # slightly more epochs for more data
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    eval_strategy="epoch",
    learning_rate=2e-5,
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = (predictions == labels).mean()
    return {"accuracy": acc}

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)

trainer.train()

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:wandb: You chose "Don't visualize my results"


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss,Accuracy
1,No log,0.743813,0.780488
2,No log,0.421343,1.000000
3,No log,0.258018,0.975610
4,No log,0.171014,1.000000
5,No log,0.148588,1.000000


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


TrainOutput(global_step=80, training_loss=0.4512182712554932, metrics={'train_runtime': 2063.9589, 'train_samples_per_second': 0.298, 'train_steps_per_second': 0.039, 'total_flos': 80907375951360.0, 'train_loss': 0.4512182712554932, 'epoch': 5.0})

## 4. Evaluate

In [13]:
eval_results = trainer.evaluate()
print(eval_results)

predictions = trainer.predict(test_dataset)
pred_labels = np.argmax(predictions.predictions, axis=1)
true_labels = predictions.label_ids

print("\nClassification Report:")
print(classification_report(true_labels, pred_labels, target_names=["low", "medium", "high"]))

{'eval_loss': 0.14858847856521606, 'eval_accuracy': 1.0, 'eval_runtime': 33.7241, 'eval_samples_per_second': 1.216, 'eval_steps_per_second': 0.089, 'epoch': 5.0}

Classification Report:
              precision    recall  f1-score   support

         low       1.00      1.00      1.00         9
      medium       1.00      1.00      1.00        12
        high       1.00      1.00      1.00        20

    accuracy                           1.00        41
   macro avg       1.00      1.00      1.00        41
weighted avg       1.00      1.00      1.00        41



## 5. Inference examples

In [14]:
from transformers import pipeline

classifier = pipeline("text-classification", model=model, tokenizer=tokenizer, device=-1)

examples = [
    "Customer: You are wasting my time! Fix this NOW!\nAgent: Ma'am, please calm down...",
    "Customer: Thanks for the quick help!\nAgent: My pleasure! Anything else?",
    "Customer: It's working now actually.\nAgent: Well then, anything is possible...",
    "Customer: Thanks a lot, It's working\nAgent: Welcome bro!",
    "Customer: Problem solved.\nAgent: Great! Want our insurance add-on?",
    "Customer: All good now.\nAgent: Awesome, cheers!"
]

for ex in examples:
    result = classifier(ex, truncation=True, max_length=256)[0]
    print(f"\nText:\n{ex}\n→ {result['label']} (score: {result['score']:.3f})")

Device set to use cpu



Text:
Customer: You are wasting my time! Fix this NOW!
Agent: Ma'am, please calm down...
→ low (score: 0.770)

Text:
Customer: Thanks for the quick help!
Agent: My pleasure! Anything else?
→ low (score: 0.617)

Text:
Customer: It's working now actually.
Agent: Well then, anything is possible...
→ low (score: 0.795)

Text:
Customer: Thanks a lot, It's working
Agent: Welcome bro!
→ high (score: 0.881)

Text:
Customer: Problem solved.
Agent: Great! Want our insurance add-on?
→ low (score: 0.643)

Text:
Customer: All good now.
Agent: Awesome, cheers!
→ high (score: 0.926)
